# 02 Data Preparation

Ziel dieses Notebooks ist es, die drei Freitag-Dateien des CICIDS2017-Datensatzes zusammenzuführen, relevante Modellfeatures auszuwählen, fehlerhafte Werte zu bereinigen und einen finalen Datensatz für das Modelltraining zu speichern.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_DATA_PATH = Path("../data/raw")
PROCESSED_DATA_PATH = Path("../data/processed")

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

files = {
    "friday_morning": RAW_DATA_PATH / "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "friday_portscan": RAW_DATA_PATH / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "friday_ddos": RAW_DATA_PATH / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
}

In [3]:
dfs = {}

for name, path in files.items():
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    dfs[name] = df
    print(f"{name}: {df.shape}")

friday_morning: (191033, 85)
friday_portscan: (286467, 85)
friday_ddos: (225745, 85)


Die drei Freitag-Dateien werden erneut geladen und die Spaltennamen werden vereinheitlicht, indem überflüssige Leerzeichen entfernt werden.

In [4]:
df_all = pd.concat(dfs.values(), ignore_index=True)

print("Zusammengeführter Datensatz:", df_all.shape)
print(df_all["Label"].value_counts())

Zusammengeführter Datensatz: (703245, 85)
Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64


Die drei Dateien werden zeilenweise zu einem gemeinsamen Datensatz zusammengeführt. Dadurch entsteht ein gemeinsamer Datenbestand mit normalem Netzwerkverkehr sowie den Angriffstypen Bot, PortScan und DDoS.

In [5]:
df_all["target"] = df_all["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

print(df_all["target"].value_counts())

target
0    414322
1    288923
Name: count, dtype: int64


Für die Modellierung wird eine binäre Zielvariable erstellt. BENIGN wird als normaler Netzwerkverkehr mit 0 codiert. Alle anderen Labels werden als Angriff mit 1 codiert.

In [6]:
columns_to_drop = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]

df_model = df_all.drop(columns=[col for col in columns_to_drop if col in df_all.columns])

print("Datensatz nach Entfernen irrelevanter Spalten:", df_model.shape)
print(df_model.columns.tolist())

Datensatz nach Entfernen irrelevanter Spalten: (703245, 78)
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag C

Identifikations- und Metadaten wie IP-Adressen, Ports, Protokoll, Zeitstempel und das ursprüngliche Label werden entfernt. Für das Modell bleiben nur numerische Netzwerkmerkmale sowie die binäre Zielvariable `target` erhalten.

In [7]:
df_model = df_model.replace([np.inf, -np.inf], np.nan)

print("Fehlende Werte vor Bereinigung:", df_model.isna().sum().sum())

df_model = df_model.dropna()

print("Datensatz nach Bereinigung:", df_model.shape)
print("Fehlende Werte nach Bereinigung:", df_model.isna().sum().sum())

Fehlende Werte vor Bereinigung: 1054
Datensatz nach Bereinigung: (702718, 78)
Fehlende Werte nach Bereinigung: 0


Infinity-Werte werden zunächst in NaN-Werte umgewandelt. Anschließend werden Zeilen mit fehlenden Werten entfernt, da diese beim Modelltraining zu Fehlern führen können.

In [8]:
print(df_model["target"].value_counts())
print(df_model["target"].value_counts(normalize=True))

target
0    413933
1    288785
Name: count, dtype: int64
target
0    0.589046
1    0.410954
Name: proportion, dtype: float64


In [9]:
output_file = PROCESSED_DATA_PATH / "cicids2017_friday_binary_clean.csv"

df_model.to_csv(output_file, index=False)

print(f"Finaler Datensatz gespeichert unter: {output_file}")
print("Finale Form:", df_model.shape)

Finaler Datensatz gespeichert unter: ..\data\processed\cicids2017_friday_binary_clean.csv
Finale Form: (702718, 78)


In [10]:
# Kompakte Übersicht zum zusammengeführten Datensatz

print("Datensatz-Übersicht")
print("-------------------")
print(f"Anzahl Zeilen: {df_all.shape[0]}")
print(f"Anzahl Spalten: {df_all.shape[1]}")

print("\nSpeicherverbrauch:")
memory_mb = df_all.memory_usage(deep=True).sum() / 1024**2
print(f"{memory_mb:.2f} MB")

print("\nDatentypen:")
print(df_all.dtypes.value_counts())

print("\nNicht-numerische Spalten:")
print(df_all.select_dtypes(exclude=[np.number]).columns.tolist())

print("\nLabel-Verteilung:")
print(df_all["Label"].value_counts())

print("\nFehlende Werte gesamt:")
print(df_all.isna().sum().sum())

Datensatz-Übersicht
-------------------
Anzahl Zeilen: 703245
Anzahl Spalten: 86

Speicherverbrauch:
652.80 MB

Datentypen:
int64      44
float64    37
str         5
Name: count, dtype: int64

Nicht-numerische Spalten:
['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Label']

Label-Verteilung:
Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64

Fehlende Werte gesamt:
47


## Zusammenfassung Data Preparation

In diesem Notebook wurden die drei Freitag-Dateien des CICIDS2017-Datensatzes zusammengeführt und für die Modellierung vorbereitet. Die ursprünglichen Labels wurden zu einer binären Zielvariable transformiert: BENIGN = normaler Netzwerkverkehr und alle anderen Labels = Angriff. Nicht relevante Metadaten wie IP-Adressen, Ports, Zeitstempel und das ursprüngliche Label wurden entfernt. Fehlerhafte NaN- und Infinity-Werte wurden bereinigt. Der finale bereinigte Datensatz wurde als `cicids2017_friday_binary_clean.csv` gespeichert und kann im nächsten Schritt für das Modelltraining verwendet werden.